# Yancc vs SFINCS Comparison - Part 4: Radial Ambipolar Scan with I/O - Output Update

This notebook performs a radial scan over 5 points, calculating the ambipolar radial electric field ($E_r$) at each point.
It demonstrates the new I/O capabilities, saving options, inputs, and results to a structured run directory.

In [ ]:
import numpy as np
import logging
import desc
import yancc
from pathlib import Path
from yancc.velocity_grids import MaxwellSpeedGrid, UniformPitchAngleGrid
from yancc.species import GlobalMaxwellian
from yancc.yancctools import sfincstools, yancctools, yancctools_plot, yancctools_io

# Configure logging to see scan progress
logging.basicConfig(level=logging.INFO, force=True)

### 1. Define Options and Configuration

We define the configuration dictionary early to ensure all parameters are tracked.

In [ ]:
options = {
    "output_path": "/u/npablant/analysis/yancc/runs/",
    "erho_min": -15000.0,   # Coarse scan range [V/m]
    "erho_max": 15000.0,
    "erho_num": 11,         # Points in coarse scan
    "nt": 17,               # Poloidal resolution
    "nz": 33,               # Toroidal resolution
    "rtol": 1e-4
}

### 2. Load and Scale Kinetic Profiles

We load the polynomial coefficients from the SFINCS `profiles` file.

In [ ]:
target_file = '/u/npablant/analysis/w7x/171207006/stelltran/run07/sfincs/t2.2273/profiles'

# Generate the raw polynomial functions (dimensionless coefficients)
funcs = sfincstools.generate_profile_functions(target_file)

# Define scaled functions for yancc (SFINCS profiles are in 10^20 m^-3 and keV)
# Species 1: Electrons, Species 2: Hydrogen
density_e = lambda r: funcs['species_1_density'](r) * 1e20
temp_e    = lambda r: funcs['species_1_temperature'](r) * 1e3

density_H = lambda r: funcs['species_2_density'](r) * 1e20
temp_H    = lambda r: funcs['species_2_temperature'](r) * 1e3

# Define global species objects
global_species = [
    GlobalMaxwellian(yancc.species.Electron, temperature=temp_e, density=density_e),
    GlobalMaxwellian(yancc.species.Hydrogen, temperature=temp_H, density=density_H)
]

print("Profiles loaded and species defined.")

### 3. Setup Equilibrium and Grids

In [ ]:
# Load W7-X equilibrium
eq = desc.examples.get("W7-X")

# Define radial grid (15 points)
rho_grid = np.linspace(0.2, 0.8, 5)

# Define velocity grids
speedgrid = MaxwellSpeedGrid(nx=5)
pitchgrid = UniformPitchAngleGrid(nxi=65)

### 4. Execute Radial Scan with Ambipolar Er Search

In [ ]:
results = yancctools.scan_ambipolar_profile(
    rho_grid,
    eq_type="desc",
    eq_data=eq,
    pitchgrid=pitchgrid,
    speedgrid=speedgrid,
    global_species=global_species,
    options=options
)

runid = results['runid']
print(f"Radial scan completed. RunID: {runid}")

### 5. Save Options, Inputs, and Results

We create a dedicated run directory and save all relevant data using the new `yancctools_io` module.

In [ ]:
# Determine run directory
run_dir = Path(options["output_path"]) / runid
print(f"Saving run data to: {run_dir}")

# 1. Save Options (JSON and YAML)
yancctools_io.save_options(options, run_dir)

# 2. Save Inputs (HDF5)
# We construct a dictionary of input parameters useful for reproduction/plotting
inputs = {
    "rho_grid": rho_grid,
    "profiles": {
        "rho": rho_grid,
        "Te": [temp_e(r) for r in rho_grid],
        "ne": [density_e(r) for r in rho_grid],
        "Ti": [temp_H(r) for r in rho_grid],
        "ni": [density_H(r) for r in rho_grid],
    },
    "grids": {
        "pitch_nxi": pitchgrid.nxi,
        "speed_nx": speedgrid.nx
    },
    "equilibrium": "W7-X (DESC example)",
    "source_profile_file": target_file
}
yancctools_io.save_inputs(inputs, run_dir)

# 3. Save Results (HDF5)
yancctools_io.save_results(results, run_dir)

### 6. Validation: Reload and Plot

To verify the I/O pipeline, we clear the `results` from memory and reload them from the disk before plotting.

In [ ]:
# Clear results from memory
del results
print("Results cleared from memory.")

# Reload results
loaded_results = yancctools_io.load_results(run_dir / "results.h5")
print("Results reloaded from disk.")

# Plot using loaded results
yancctools_plot.plot_ambipolar_summary(loaded_results, global_species)